# LLM Request Triage: Tests and Synthetic Baselines

This notebook installs the shared project code, runs the test suite, executes all deterministic baselines on the development-only synthetic executor, and displays the generated metrics. The synthetic executor validates the pipeline; it is not evidence for the final project conclusions.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/xfgao210/llm-request-triage.git"
BRANCH = "evaluation-engine"  # Change to main after the PR is merged.

if Path("/content").exists():
    repo = Path("/content/llm-request-triage")
    if not repo.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
else:
    repo = Path.cwd()
    if repo.name == "notebooks":
        repo = repo.parent

os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)
print(f"Repository ready at {repo}")

## Run unit and integration tests

In [ ]:
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    check=True,
)

## Run the three deterministic baselines

In [ ]:
output_dir = repo / "results" / "colab_synthetic"
subprocess.run(
    [
        sys.executable,
        "experiments/run_baselines.py",
        "--seed", "42",
        "--duration", "30",
        "--requests", "60",
        "--output-dir", str(output_dir),
    ],
    check=True,
)

## Inspect and plot the results

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

summary = pd.read_csv(output_dir / "baseline_summary.csv")
display(summary[[
    "policy", "slo_attainment", "window_throughput_rps",
    "window_goodput_rps", "drop_rate", "p95_ttft_s"
]])

figure, axes = plt.subplots(1, 2, figsize=(12, 4))
summary.plot.bar(x="policy", y="slo_attainment", ylim=(0, 1), ax=axes[0], legend=False, title="SLO attainment")
summary.plot.bar(x="policy", y=["window_throughput_rps", "window_goodput_rps"], ax=axes[1], title="Throughput and goodput")
for axis in axes:
    axis.set_xlabel("")
    axis.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## Next integration step

The synthetic workload now runs through the same `SimulatorExecutor`, scheduler contracts, lifecycle events, metrics, and reporting layers used by BurstGPT.